# Tracking Analysis Example


## Experiment Notes

**Overview:**   
**Rationale:**   
**Date of Experiment:**   
**Genotypes:**   
**Sex:**   
**Light Paradigm:**   
**Design:**   
**Misc:**   


## Imports


In [ ]:
import os
os.chdir('../')

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

from Experiment import Experiment, batch_analyze


---
## Single Experiment Analysis

Point `Experiment` at the project directory — the folder containing `tracking_config.yaml`
and a `data/` subfolder with the DTrack `.xlsx` and `.csv` files.

```
project_directory/
    tracking_config.yaml      ← tracking type, rig, experimental design
    data/
        ExperimentName.xlsx
        ExperimentName_Data_1.csv  ...
    analysis/                 ← created automatically
    qc/                       ← created automatically
```

All parameters (rig calibration, tracking type, facet cutoffs) are read from `tracking_config.yaml`
automatically — no manual `Parameters` object needed.


In [ ]:
# Load the experiment — reads tracking_config.yaml and all data files
exp = Experiment('./Data/')
print(exp)


### Experiment Summary

Prints a detailed overview of the experiment configuration, parameters, and per-tracker
data quality. Saves a `.txt` copy to `analysis/`.


In [ ]:
exp.experiment_summary();


### Quality Control

Prints the data quality report (% high-quality, not-found, and indiscernible frames per tracker)
and flags any trackers below the quality threshold. Saves a CSV to `qc/`.


In [ ]:
exp.qc(cutoff=0.9);


### Individual Plots

Each plot method is appropriate to the tracking type defined in `tracking_config.yaml`.
For a `TWOCHOICETRACKER` the relevant plots are PI, percentage, transitions, and total distance.
The `_facet` variants split the recording into phases defined by `facet_cutoffs` in the YAML.


In [ ]:
# Preference index — full recording
exp.arena.plot_pi()


In [ ]:
# Preference index — split by facet cutoffs from tracking_config.yaml
exp.arena.plot_pi_facet(cutoffs=exp.facet_cutoffs)


In [ ]:
# Choice percentage — full recording
exp.arena.plot_percentage()


In [ ]:
# Choice percentage — split by facet cutoffs
exp.arena.plot_percentage_facet(cutoffs=exp.facet_cutoffs)


In [ ]:
# Transitions per minute — full recording
exp.arena.plot_transitions()


In [ ]:
# Transitions per minute — split by facet cutoffs
exp.arena.plot_transitions_facet(cutoffs=exp.facet_cutoffs)


In [ ]:
# Total distance per minute — split by facet cutoffs
exp.arena.plot_totaldistance_facet(cutoffs=exp.facet_cutoffs)


### Summary Data & Statistics

Save summary CSVs (flat and faceted) and run pairwise comparisons across treatment groups.
All outputs go to `analysis/`.


In [ ]:
# Save flat and faceted summary CSVs
summary, summary_facet = exp.save_summary()
summary.head()


In [ ]:
# Faceted summary (one row per tracker per phase)
summary_facet.head(12)


In [ ]:
# Pairwise statistics across facet phases — saved to analysis/
exp.stats();


### PDF Report

Generates a multi-page PDF in `analysis/` containing:
- Experiment summary text
- Data quality table (colour-coded pass/fail)
- Per-tracker QC position grids
- All analysis plots appropriate for the tracking type


In [ ]:
exp.create_report();


### Full Pipeline (one call)

`run_analysis()` runs all steps in sequence — experiment summary, QC, summary CSVs,
all plots, and statistics — and saves everything to `analysis/` and `qc/`.
Use this instead of the individual steps above when you don't need interactive exploration.


In [ ]:
exp.run_analysis();


---
## Batch Analysis

`batch_analyze()` scans every immediate subdirectory of a parent folder, identifies valid
experiment directories (those with `tracking_config.yaml` and a `data/*.xlsx`), and runs
`run_analysis()` + `create_report()` on each one automatically.

Expected parent layout:
```
parent_directory/
    Experiment_A/
        tracking_config.yaml
        data/
            ExperimentA.xlsx
            ExperimentA_Data_1.csv  ...
    Experiment_B/
        tracking_config.yaml
        data/
            ExperimentB.xlsx  ...
    ...
```

Each experiment's outputs are saved inside its own `analysis/` and `qc/` subdirectories.
The function returns a dict of `{path: 'ok' | error_message}` so failures can be inspected.


In [ ]:
# Run analysis on every experiment found under the parent directory.
# Replace the path below with your actual parent directory.
results = batch_analyze("./Data/")


In [ ]:
# Inspect results — 'ok' means success, anything else is an error message
for path, status in results.items():
    print(f"{'OK' if status == 'ok' else 'FAIL':4s}  {path}")
    if status != 'ok':
        print(f"      {status}")
